# NORTHSTAR practical — first contact

**DASC513 · Data Engineering block · Week 5 · foundations**

> ### Before you do anything else
> **File → Save a copy in Drive.** Until you do that you are looking at a read-only
> copy and your work disappears when the tab closes. Do it now, then carry on.

This notebook runs in Google Colab, or on your own machine if you have Python and
pandas. Nothing needs installing. The data loads in the next cell straight from
the course GitHub repository; if that fails (no internet, or you are working
offline), it will ask you to upload the CSV files you downloaded from Canvas.

## How to use this notebook

A notebook is a sequence of **cells**. Grey cells are code; white cells are
writing. Run a cell by clicking into it and pressing **Shift + Enter**.

Three kinds of cell in this notebook:

| Marker | What to do |
|---|---|
| **READ** | Explanation. No code to run. |
| **RUN** | Code that is already written. Read it first, predict what it will do, then run it. |
| **YOUR TURN** | Code with something missing, or a written answer to give. |

Two rules that will save you an hour of confusion:

1. **Cells run in the order you run them, not the order they appear.** If
   something breaks unexpectedly, go to *Kernel → Restart and Run All* and start
   again from the top.
2. **Read code before running it.** Say it out loud in English. `df["sex"]`
   is "the sex column of df". `df["sex"].value_counts()` is "count how many
   times each value appears in the sex column of df". If you can say it, you
   understand it.

## The enquiry loop

Every task below follows the same five steps. This is the actual method; the
pandas is just how you carry it out.

1. **Observe** — look at the data without deciding anything yet.
2. **Question** — turn what you noticed into a question that has an answer.
3. **Test** — write the smallest piece of code that answers it.
4. **Interpret** — what does the answer mean *for the research question*?
5. **Decide** — fix it, flag it, exclude it, or accept it. Write down which and why.

Step 5 is the one people skip. A data quality report full of tables and empty of
decisions is not a data quality report.

---
# Part 0 — Opening the data

**READ.** `pandas` is the library everyone uses for table-shaped data in Python. By convention it is imported as `pd`. A table in pandas is called a **DataFrame**, and almost everyone calls their first one `df`.

In [ ]:
# RUN. Load the data. Works in Colab, and on your own machine if the CSV files
# are beside the notebook. If neither, Colab will ask you to upload them.
import io, os
import pandas as pd

DATA_REPO = "https://raw.githubusercontent.com/civic-health-innovation-labs/dasc513-data-engineering/main/"
DATA_URL = DATA_REPO + "NORTHSTAR_redcap_export.csv"
DICT_URL = DATA_REPO + "NORTHSTAR_data_dictionary.csv"

_uploaded = {}

def load(name, url):
    if url:
        try:
            return pd.read_csv(url, dtype=str)
        except Exception as e:          # offline, or the repository moved
            print(f"Could not fetch {name} from GitHub ({type(e).__name__}); trying a local copy.")
    if os.path.exists(name):
        return pd.read_csv(name, dtype=str)
    if name not in _uploaded:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f"{name} not found. Put it next to this notebook, or check the URL above.")
        print(f"Choose {name} (from Canvas) when the button appears.")
        _uploaded.update(files.upload())
    return pd.read_csv(io.BytesIO(_uploaded[name]), dtype=str)

df = load("NORTHSTAR_redcap_export.csv", DATA_URL)
data_dictionary = load("NORTHSTAR_data_dictionary.csv", DICT_URL)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"loaded {df.shape[0]} records with {df.shape[1]} fields, and a data dictionary of {len(data_dictionary)} rows")

**READ — why `dtype=str`?**

Left alone, pandas guesses what each column contains. It is quite good at
guessing and that is exactly the problem: it will quietly turn `"3.7"` into the
number 3.7 and `"01"` into the number 1, and it will guess wrong on at least one
of the date columns in this file.

Today we want to see the data **as it was actually entered**, so we read
everything as text and convert later, deliberately, when we have decided what
each column should be.

> **Challenge to think about (discussion, not code):** what is the risk of the
> opposite habit — always letting the tool convert types for you? Where would
> that bite in a real pipeline?

In [ ]:
# RUN. head() shows the first few rows. The .T flips rows and columns, which is
# much easier to read when a table is this wide.
df.head(3).T

In [ ]:
# RUN. The data dictionary is a table too. This shows the first few rows.
# Later you will want:  data_dictionary[data_dictionary["variable_name"] == "some_field"]
data_dictionary.head(8)

**YOUR TURN 1.**

Look at the output above and answer in the cell below (in words, not code):

- What does one **row** represent?
- Name three columns that could identify a real person.
- Name one column you do not understand at all.

Then check your third answer against `NORTHSTAR_data_dictionary.csv`. If it is
not in there, that is not your mistake — write down that the dictionary is
incomplete, because that is itself a finding.

*Your answer here (double-click to edit this cell):*

- A row represents…
- Possible identifiers:
- I do not understand:

---
# Part 1 — Asking a single column a question

**READ.** The single most useful thing you can do to a column is count what is
in it. `value_counts()` does that.

In [ ]:
# RUN. Predict the answer before you run it: how many different values
# do you expect the sex column to contain?
df["sex"].value_counts(dropna=False)

**Interpret.** You probably expected two values. Count them.

This is a **conformance** problem: the values do not conform to the format the
data dictionary specifies. It happens because the CRF used a free-text box where
it should have used a dropdown, and because three sites had different research
nurses typing into it.

**Why it matters:** any code that does `df[df["sex"] == "F"]` silently drops
patients. Not with an error. Silently.

In [ ]:
# YOUR TURN 2. Pick a different column and count its values.
# Try: "consent_type", "outcome", "smoking_status", "ethnic_category", "site_code"
# Change the column name below and run it.

df["consent_type"].value_counts(dropna=False)

**YOUR TURN 2, part b.** Run the cell above on `ethnic_category`. You will see
both `99` and `999`.

One of those is a real code meaning "not known" (it is in the NHS Data
Dictionary). The other is somebody's shorthand for "I didn't fill this in".

- Which is which? How would you find out?
- What happens to your analysis if you treat them as the same thing? What
  happens if you treat `99` as a real ethnic group?

---
# Part 2 — Three fields you will map to a standard

**READ.** Later in the session, on paper, you will take three fields and say
which terminology each should map to, roughly which concept, and what is lost
in the mapping. The cells below show you what is actually in those fields, so
the mapping exercise is done against the real values rather than the column
name.

Three fields, three kinds of problem:

- `com_other_txt` — a free-text "other comorbidity" box
- `meds_on_admission_txt` — medication as transcribed from the drug chart
- `crp_adm` — a laboratory result, C-reactive protein at admission

In [ ]:
# RUN. Every distinct value in the free-text comorbidity field, with a count.
df["com_other_txt"].value_counts(dropna=False)

In [ ]:
# RUN. Ten medication strings. Read them as a pharmacist would, then as a
# computer would.
for m in df["meds_on_admission_txt"].dropna().head(10):
    print("-", m)

In [ ]:
# RUN. The lab result: what does the data dictionary say about it, and what
# do the first few values look like?
print(data_dictionary[data_dictionary["variable_name"] == "crp_adm"].T)
print()
print("First eight values:", df["crp_adm"].head(8).tolist())

# And the values that are NOT numbers. A lab result that cannot be turned into
# a number is not a lab result yet.
as_number = pd.to_numeric(df["crp_adm"], errors="coerce")
print("Values that are not numbers:", df.loc[as_number.isna(), "crp_adm"].value_counts(dropna=False).to_dict())
print("Largest numeric value:", as_number.max())

**YOUR TURN 3.** Answer in the cell below, in words. You will transfer this to
the paper mapping sheet later.

For each of the three fields:

1. Which terminology should it map to? (The lecture gave you four: SNOMED CT,
   ICD-10, dm+d, LOINC.)
2. Give one example value and say roughly what concept it would become.
3. What information is lost in the mapping? Look at `OSA on CPAP` and at
   `see GP summary` in particular.
4. For the lab result the *code* is easy. What about the *values*? Which of
   them are not results at all?
5. Which field could not be mapped by a computer without a person deciding
   something first?

*Your answer here:*

---
# Part 3 — What the data dictionary does not tell you

**READ.** In Your Turn 1 you checked one column against the data dictionary by
eye. The computer can check all of them at once. This is the first time in the
block that you use code to answer a question about the *documentation* rather
than about the data, and it is a habit worth keeping.

In [ ]:
# RUN. Which columns in the export are not described in the data dictionary?
documented = set(data_dictionary["variable_name"])
in_export = [c for c in df.columns if not c.endswith("_complete")]

undocumented = [c for c in in_export if c not in documented]

print(f"{len(in_export)} fields in the export, {len(documented)} described in the dictionary")
print(f"{len(undocumented)} fields with no entry:\n")
for c in undocumented:
    print("-", c)

**YOUR TURN 4.** Look at the list.

1. Which undocumented field worries you most, and why? (Think about what it
   might contain, not just its name.)
2. If you were writing the provenance sheet, which box does this finding belong
   in?
3. Is an incomplete data dictionary a data quality problem, a governance
   problem, or both?

*Your answer here:*

---
# Part 4 — Before you leave the notebook

The rest of the session is on paper: the intended-use statement, the provenance
sheet, and the FAIR checklist. Everything you noticed here feeds into them.

One sentence, in the cell below, that you will hand in as your exit ticket:

> The thing I most want to check about this dataset before I would trust it is…

*Your answer here:*